In [1]:
import numpy as np 
import pandas as pd 
import math 
from collections import defaultdict

In [8]:
class MixedNaiveBayes: 
    def __init__(self , categorical_indices = None , alpha = 1.0): 
        """ 
        categorical_indices : list of dices of categorical features.
        alpha : Laplace smoothing paramter.
        """
        self.categorical_indices = categorical_indices if categorical_indices else []
        self.alpha = alpha

    def fit(self , X , y): 
        # convert X and y into numpy array 
        X , y = np.asarray(X) , np.asarray(y)

        # find the unique classes
        self.classes_ = np.unique(y)
        # find the count of features 
        n_features = X.shape[1]

        # split numerical and categorical data 
        self.numerical_indices = [i for i in range(n_features) if i not in self.categorical_indices]

        # find the priors 
        self.priors_ = {c: np.mean(y == c) for c in self.classes_}
        # store params.(mean and variance for every class of each numerical features)
        self.means_ = defaultdict(dict)
        self.var_ = defaultdict(dict)

        # self.cat_counts_[class_c][feature_index][feature_value] = count
        self.cat_counts_ = defaultdict(lambda: defaultdict(lambda: defaultdict(int))) 
        # self.cat_totals_[class_c][feature_index] = total_count 
        self.cat_total_ = defaultdict(lambda: defaultdict(int))
        self.cat_uniques = defaultdict(set)

        # now go through each class 
        for c in self.classes_: 
            Xc = X[y == c]

            # numerical features -> store mean and variance
            for j in self.numerical_indices: 
                self.means_[c][j] = Xc[ : , j].astype(float).mean()
                self.var_[c][j] = Xc[ : , j].astype(float).var() + 1e-9 # adding epsilon to avoid zero division

            # for categorical features -> store counts 
            for j in self.categorical_indices: 
                # different values of j-th categorical column 
                values , counts = np.unique(Xc[ : , j] , return_counts = True)

                for v , cnt in zip(values , counts): 
                    self.cat_counts_[c][j][v] = cnt
                # save total samples for this column 
                self.cat_total_[c][j] = len(Xc[ : , j])
                self.cat_uniques[j].update(np.unique(X[ : , j]))
        
        return self

    def _gaussian_log_prob(self, x, mean, var):
        """ log of Gaussian PDF """
        return -0.5 * math.log(2 * math.pi * var) - ((x - mean) ** 2) / (2 * var) 

    def _categorical_log_prob(self , value , class_c , j): 
        """ log probability with laplace smoothing """
        cnt = self.cat_counts_[class_c][j].get(value , 0)
        total = self.cat_total_[class_c][j]

        n_vals = len(self.cat_uniques[j])
        prob = (cnt + self.alpha) / (total + self.alpha * n_vals) 
        return math.log(prob)

    def predict_row(self , x_row): 
        """ predict class for a single row and return the detailed terms """
        log_posteriors = {}
        details = {}

        for c in self.classes_: 
            log_prior = math.log(self.priors_[c])

            # numerical features
            num_terms = sum(
                self._gaussian_log_prob(float(x_row[j]) , self.means_[c][j] , self.var_[c][j]) for j in self.numerical_indices
            )

            # categorical features
            cat_terms = sum(
                self._categorical_log_prob(x_row[j], c, j) for j in self.categorical_indices
            )

            total_log = log_prior + num_terms + cat_terms

            log_posteriors[c] = total_log
            details[c] = {
                "log_prior": log_prior,
                "numerical": num_terms,
                "categorical": cat_terms,
                "log_posterior": total_log
            }
        # pick class with max log posterior
        predicted_class = max(log_posteriors, key = log_posteriors.get)
        return predicted_class, details

    def predict(self , X): 
        return np.array([self.predict_row(row)[0] for row in np.asarray(X)])

In [9]:
data = np.array([
    [5.1, 3.5, 'Red'],
    [6.0, 3.0, 'Blue'],
    [5.5, 3.2, 'Red'],
    [6.2, 3.4, 'Blue'],
    [5.0, 3.6, 'Red']
])
labels = np.array(['Yes', 'No', 'Yes', 'No', 'Yes'])

In [10]:
nb = MixedNaiveBayes(categorical_indices = [2])  # 3rd column is categorical
nb.fit(data, labels)

In [11]:
# Predict single row
x_test = [5.4, 3.3, 'Red']
pred, details = nb.predict_row(x_test)

In [12]:
print("Predicted class:", pred)
print("Details per class:")
for c, d in details.items():
    print(c, d)

Predicted class: Yes
Details per class:
No {'log_prior': -0.916290731874155, 'numerical': np.float64(-22.550851670356348), 'categorical': -1.3862943611198906, 'log_posterior': np.float64(-24.853436763350395)}
Yes {'log_prior': -0.5108256237659907, 'numerical': np.float64(0.7303708743113841), 'categorical': -0.2231435513142097, 'log_posterior': np.float64(-0.003598300768816287)}


In [13]:
# Predict multiple
print("Batch prediction:", nb.predict([
    [6.1, 3.1, 'Blue'],
    [5.2, 3.4, 'Red']
]))

Batch prediction: ['No' 'Yes']
